# Lab 07: Ratings Systems

This lab walks you through four popular rating systems used in sports analytics: **Elo**, **Massey**, **PageRank**, and **Glicko**. Each captures team strength from a different angle. By the end you will:

- Understand what a rating system is and when to use each method
- Implement Elo ratings with configurable K-factor
- Compute Massey ratings via linear algebra
- Apply PageRank to game results as a directed graph
- Track uncertainty with Glicko's rating deviation (RD)
- Compare all four systems side by side
- Backtest each system on historical (or synthetic) game data

---

## Prerequisites

- **Labs 01-06 completed** — you understand the SportsQuant data pipeline
- **Linear algebra basics** — matrix inversion, solving Ax = b
- **Probability basics** — Bayes' theorem, normal distributions

### Rating Systems Overview

| System | Core Idea | Best For | Output |
|---|---|---|---|
| **Elo** | Update after each game | Chess, 1v1 sports | Single rating per team |
| **Massey** | Solve linear system | League seasons | Offensive + defensive ratings |
| **PageRank** | Transitive wins | College sports, polls | Single rating per team |
| **Glicko** | Elo + uncertainty | Small samples, new teams | Rating ± RD per team |

---

## Section 1: Setup — Imports and Synthetic Data

We import the SportsQuant rating modules and set up synthetic game data for when the database doesn't have enough results yet. All four rating systems will operate on the same game dataset so we can compare fairly.

In [ ]:
# Cell 4: Core imports
import math
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sportsquant.models.ratings.massey_ratings import MasseyRatings, MasseyRatingsConfig
from sportsquant.models.ratings.pagerank_ratings import PageRankRatings

# Enable nested event loops in Jupyter
import nest_asyncio
nest_asyncio.apply()

print("Imports loaded successfully.")

In [ ]:
# Cell 5: Generate synthetic game results
#
# We simulate a 10-team league playing a round-robin schedule.
# Each team has a latent "true strength" that determines win probability.

np.random.seed(42)

TEAMS = [
    "KC", "BUF", "SF", "PHI", "DET",
    "DAL", "BAL", "MIA", "CLE", "CAR",
]

# True strengths (higher = better team)
TRUE_STRENGTH = {
    "KC": 10.0, "BUF": 8.5, "SF": 7.0, "PHI": 6.5, "DET": 6.0,
    "DAL": 4.0, "BAL": 5.5, "MIA": 3.5, "CLE": 2.0, "CAR": 1.0,
}

HOME_ADVANTAGE = 3.0  # points

def generate_games(n_games: int = 200, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic game results with realistic score distributions."""
    rng = np.random.default_rng(seed)
    records = []
    for _ in range(n_games):
        home = rng.choice(TEAMS)
        away = rng.choice([t for t in TEAMS if t != home])
        home_strength = TRUE_STRENGTH[home] + HOME_ADVANTAGE
        away_strength = TRUE_STRENGTH[away]
        margin = home_strength - away_strength + rng.normal(0, 4)
        home_score = max(7, int(22 + away_strength / 2 + rng.normal(0, 3)))
        away_score = max(7, int(22 + away_strength / 2 - margin / 2 + rng.normal(0, 3)))
        # Adjust scores to match margin
        actual_margin = home_score - away_score
        away_score += int(round(actual_margin - margin))
        away_score = max(7, away_score)
        winner = home if home_score > away_score else away
        loser = away if winner == home else home
        records.append({
            "HOME_TEAM": home,
            "AWAY_TEAM": away,
            "HOME_SCORE": home_score,
            "AWAY_SCORE": away_score,
            "WINNER": winner,
            "LOSER": loser,
            "WINNER_SCORE": max(home_score, away_score),
            "LOSER_SCORE": min(home_score, away_score),
        })
    return pd.DataFrame(records)

games_df = generate_games(n_games=200)
print(f"Generated {len(games_df)} synthetic games")
print(f"\nSample games:")
print(games_df.head(10).to_string(index=False))
print(f"\nWin distribution:")
print(games_df["WINNER"].value_counts().sort_index())

---

## Section 2: What Is a Rating System?

A **rating system** assigns a numerical value to each team that represents its strength. Higher ratings mean stronger teams. The goal is to produce ratings that are **predictive** — if Team A has a higher rating than Team B, Team A should win more often than not.

### Four Approaches

1. **Elo** — Iterative update after each game. Simple, well-known, and widely used (chess, FIFA rankings). The K-factor controls how much each game moves the rating.

2. **Massey** — Solve a linear system `M·r = p` where `M` encodes who played whom and `p` encodes point margins. Decomposes into offensive/defensive components.

3. **PageRank** — Treat games as a directed graph where wins are edges. Iteratively propagate strength through the network. A win over a strong team counts more than a win over a weak team.

4. **Glicko** — Extends Elo by adding a **rating deviation (RD)** that represents uncertainty. New teams start with high RD; as they play more games, RD shrinks. This is crucial for small-sample situations.

### When to Use What

| Situation | Best System | Why |
|---|---|---|
| Head-to-head, sequential games | Elo | Simple, well-calibrated |
| Full season, need offense/defense split | Massey | Decomposes ratings |
| Transitive strength (A beat B beat C) | PageRank | Captures win quality |
| New teams, small samples | Glicko | Uncertainty quantification |

---

## Section 3: Elo Ratings

The **Elo rating system** was developed by Arpad Elo for chess. After each game, both teams' ratings are updated:

```
expected_A = 1 / (1 + 10^((rating_B - rating_A) / 400))
new_rating_A = rating_A + K * (outcome_A - expected_A)
```

- `K` is the **K-factor** (typically 20-40): higher K = more reactive to recent games
- `outcome_A` is 1 for win, 0 for loss, 0.5 for draw
- `expected_A` is the predicted win probability based on rating difference

The 400 in the formula means a **400-point rating difference** corresponds to a 10:1 expected win ratio (≈91% win probability).

In [ ]:
# Cell 8: Elo rating system implementation

@dataclass
class EloRating:
    """Simple Elo rating system.

    Attributes:
        K: K-factor controlling rating volatility.
        initial_rating: Starting rating for new teams.
        home_advantage: Home field advantage in Elo points.
    """
    K: float = 32.0
    initial_rating: float = 1500.0
    home_advantage: float = 65.0  # ~3 points scaled to Elo
    ratings: dict = field(default_factory=dict)
    history: list = field(default_factory=list)

    def get_rating(self, team: str) -> float:
        """Get current rating for a team."""
        return self.ratings.get(team, self.initial_rating)

    def expected_score(self, rating_a: float, rating_b: float) -> float:
        """Calculate expected score (win probability) for team A."""
        return 1.0 / (1.0 + 10.0 ** ((rating_b - rating_a) / 400.0))

    def update(self, home: str, away: str, home_score: int, away_score: int) -> None:
        """Update ratings after a game."""
        rating_home = self.get_rating(home) + self.home_advantage
        rating_away = self.get_rating(away)

        expected_home = self.expected_score(rating_home, rating_away)
        expected_away = 1.0 - expected_home

        if home_score > away_score:
            outcome_home = 1.0
        elif home_score < away_score:
            outcome_home = 0.0
        else:
            outcome_home = 0.5
        outcome_away = 1.0 - outcome_home

        old_home = self.get_rating(home)
        old_away = self.get_rating(away)

        self.ratings[home] = old_home + self.K * (outcome_home - expected_home)
        self.ratings[away] = old_away + self.K * (outcome_away - expected_away)

        self.history.append({
            "home": home, "away": away,
            "home_score": home_score, "away_score": away_score,
            "home_rating_before": old_home,
            "away_rating_before": old_away,
            "home_rating_after": self.ratings[home],
            "away_rating_after": self.ratings[away],
        })

    def win_probability(self, home: str, away: str) -> float:
        """Calculate home win probability."""
        r_home = self.get_rating(home) + self.home_advantage
        r_away = self.get_rating(away)
        return self.expected_score(r_home, r_away)

print("EloRating class defined.")

In [ ]:
# Cell 9: Run Elo on synthetic game data

elo = EloRating(K=32, initial_rating=1500, home_advantage=65)

for _, game in games_df.iterrows():
    elo.update(game["HOME_TEAM"], game["AWAY_TEAM"], game["HOME_SCORE"], game["AWAY_SCORE"])

print("Elo Ratings (K=32):")
print(f"{'Team':<6} {'Rating':>8} {'True':>8}")
print(f"{'-'*6} {'-'*8} {'-'*8}")
for team in sorted(TEAMS, key=lambda t: elo.get_rating(t), reverse=True):
    print(f"{team:<6} {elo.get_rating(team):>8.1f} {TRUE_STRENGTH[team]:>8.1f}")

In [ ]:
# Cell 10: Plot Elo rating evolution over games

# Track ratings over time for each team
elo_track = {team: [] for team in TEAMS}
elo_track_temp = EloRating(K=32, initial_rating=1500, home_advantage=65)

for _, game in games_df.iterrows():
    elo_track_temp.update(game["HOME_TEAM"], game["AWAY_TEAM"], game["HOME_SCORE"], game["AWAY_SCORE"])
    for team in TEAMS:
        elo_track[team].append(elo_track_temp.get_rating(team))

fig, ax = plt.subplots(figsize=(14, 7))
for team in TEAMS:
    ax.plot(elo_track[team], label=team, linewidth=1.5, alpha=0.8)

ax.set_xlabel("Game Number", fontsize=12)
ax.set_ylabel("Elo Rating", fontsize=12)
ax.set_title("Elo Rating Evolution (K=32)", fontsize=14, fontweight="bold")
ax.legend(loc="upper left", fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
ax.axhline(y=1500, color="black", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

---

## Section 4: Predicting with Elo

Elo ratings convert directly to win probabilities. A rating difference of `Δ` points gives an expected score of:

```
P(home wins) = 1 / (1 + 10^((rating_away - rating_home - HFA) / 400))
```

Let's use this to predict upcoming games and compare with actual results.

In [ ]:
# Cell 12: Elo win probability predictions
#
# For each game in our dataset, compute the Elo-predicted win probability
# and compare with actual outcomes.

elo_pred = EloRating(K=32, initial_rating=1500, home_advantage=65)
results = []

# Use first 150 games for training, last 50 for testing
train_games = games_df.iloc[:150]
test_games = games_df.iloc[150:]

for _, game in train_games.iterrows():
    elo_pred.update(game["HOME_TEAM"], game["AWAY_TEAM"], game["HOME_SCORE"], game["AWAY_SCORE"])

# Now predict on test set
correct = 0
total = 0
brier_scores = []

for _, game in test_games.iterrows():
    prob_home = elo_pred.win_probability(game["HOME_TEAM"], game["AWAY_TEAM"])
    actual_home_win = 1 if game["HOME_SCORE"] > game["AWAY_SCORE"] else 0
    predicted_home_win = 1 if prob_home > 0.5 else 0
    correct += (predicted_home_win == actual_home_win)
    total += 1
    brier_scores.append((prob_home - actual_home_win) ** 2)
    # Still update ratings after each game for continuous learning
    elo_pred.update(game["HOME_TEAM"], game["AWAY_TEAM"], game["HOME_SCORE"], game["AWAY_SCORE"])

accuracy = correct / total if total > 0 else 0
brier = np.mean(brier_scores) if brier_scores else 0

print(f"Elo Prediction Results (test set):")
print(f"  Accuracy: {accuracy:.3f} ({correct}/{total})")
print(f"  Brier Score: {brier:.4f}")
print(f"  (Brier score: lower is better, 0.25 = random guessing)")

---

## Section 5: Massey Ratings

The **Massey method** (Kenneth Massey, 1997) solves a linear system to find team ratings that best explain observed point margins. It decomposes each team's rating into **offensive** and **defensive** components.

The core equation: `M · r = p`
- `M[i,j]` = number of times team i played team j
- `p[i]` = average point margin for team i
- `r[i]` = the rating to solve for

SportsQuant's `MasseyRatings` class handles matrix construction, solving, and decomposition.

In [ ]:
# Cell 14: Compute Massey ratings using SportsQuant

massey = MasseyRatings(home_advantage=3.0)

# Massey expects a DataFrame with HOME_TEAM, AWAY_TEAM, HOME_SCORE, AWAY_SCORE
massey_ratings = massey.compute_ratings(games_df)

print("Massey Ratings:")
print(massey_ratings.sort_values("overall_rating", ascending=False).to_string())
print(f"\nRating range: {massey_ratings['overall_rating'].min():.2f} to {massey_ratings['overall_rating'].max():.2f}")
print(f"Rating std: {massey_ratings['overall_rating'].std():.2f}")

In [ ]:
# Cell 15: Massey offensive/defensive decomposition and visualization

# Decompose each team's rating into offensive and defensive components
off_def = {}
for team in massey_ratings.index:
    rating = massey_ratings.loc[team, "overall_rating"]
    off, defe = massey.decompose_rating(rating)
    off_def[team] = {"offensive": off, "defensive": defe, "overall": rating}

off_def_df = pd.DataFrame(off_def).T
print("Massey Offensive/Defensive Decomposition:")
print(off_def_df.sort_values("overall", ascending=False).to_string())

# Plot offensive vs defensive
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(off_def_df["offensive"], off_def_df["defensive"], s=100, alpha=0.7)
for team in off_def_df.index:
    ax.annotate(team, (off_def_df.loc[team, "offensive"], off_def_df.loc[team, "defensive"]),
                fontsize=11, fontweight="bold", ha="center", va="bottom")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Offensive Rating", fontsize=12)
ax.set_ylabel("Defensive Rating", fontsize=12)
ax.set_title("Massey Offensive vs Defensive Decomposition", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Section 6: PageRank Ratings

**PageRank** (the algorithm behind Google Search) can be adapted for sports ratings. The key insight: **a win over a strong team should count more than a win over a weak team**. This is the transitive property of strength.

The algorithm:
1. Build a transition matrix where edges represent wins
2. Apply power iteration with a damping factor (typically 0.85)
3. A team's rating = the steady-state probability of reaching it via wins

SportsQuant's `PageRankRatings` class implements this with margin-weighted transitions.

In [ ]:
# Cell 17: Compute PageRank ratings

pagerank = PageRankRatings(damping=0.85, max_iterations=100, tolerance=1e-6)

# PageRank expects WINNER, LOSER, WINNER_SCORE, LOSER_SCORE columns
pr_ratings = pagerank.compute_ratings(games_df, TEAMS)

# Sort by rating
pr_sorted = sorted(pr_ratings.items(), key=lambda x: x[1], reverse=True)

print("PageRank Ratings (damping=0.85):")
print(f"{'Team':<6} {'PageRank':>10} {'True':>8}")
print(f"{'-'*6} {'-'*10} {'-'*8}")
for team, rating in pr_sorted:
    print(f"{team:<6} {rating:>10.6f} {TRUE_STRENGTH[team]:>8.1f}")

In [ ]:
# Cell 18: Compare PageRank strength of schedule

sos = {}
for team in TEAMS:
    sos[team] = pagerank.strength_of_schedule(games_df, pr_ratings, team)

print("PageRank Strength of Schedule:")
print(f"{'Team':<6} {'Rating':>10} {'SoS':>10}")
print(f"{'-'*6} {'-'*10} {'-'*10}")
for team in sorted(TEAMS, key=lambda t: pr_ratings[t], reverse=True):
    print(f"{team:<6} {pr_ratings[team]:>10.6f} {sos[team]:>10.6f}")

# Plot PageRank ratings
fig, ax = plt.subplots(figsize=(10, 6))
teams_sorted = sorted(TEAMS, key=lambda t: pr_ratings[t], reverse=True)
ratings_sorted = [pr_ratings[t] for t in teams_sorted]
colors = ['#2E86AB' if r > np.mean(ratings_sorted) else '#E63946' for r in ratings_sorted]
ax.bar(teams_sorted, ratings_sorted, color=colors, alpha=0.8)
ax.axhline(y=np.mean(ratings_sorted), color='black', linestyle='--', alpha=0.5, label='Average')
ax.set_xlabel('Team', fontsize=12)
ax.set_ylabel('PageRank Rating', fontsize=12)
ax.set_title('PageRank Team Ratings', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---

## Section 7: Glicko Ratings

The **Glicko system** (Mark Glickman, 1995) extends Elo by adding a **rating deviation (RD)** that measures uncertainty in a team's rating.

Key concepts:
- **Rating (r)**: Best estimate of skill, similar to Elo
- **Rating Deviation (RD)**: Uncertainty measure (like a standard deviation)
- **Rating Volatility (σ)**: How much rating fluctuates between periods

A new team starts with `r=1500, RD=350`. After each game, RD shrinks (we learn more). Between rating periods, RD grows (we become less certain).

This is especially useful for:
- New teams with few games played
- Off-season uncertainty
- Deciding how much to trust a rating

In [ ]:
# Cell 20: Glicko rating system implementation

@dataclass
class GlickoTeam:
    """A team's Glicko rating and RD."""
    rating: float = 1500.0
    rd: float = 350.0
    vol: float = 0.06  # volatility


class GlickoRating:
    """Glicko-1 rating system.

    Implements the Glicko-1 algorithm with rating deviation tracking.
    After each rating period, RD shrinks for teams that played and
    grows for teams that didn't.
    """

    SCALE = 173.7178  # Glicko scaling factor

    def __init__(self, c: float = 63.2, initial_rating: float = 1500.0, initial_rd: float = 350.0):
        self.c = c  # RD increase per period
        self.initial_rating = initial_rating
        self.initial_rd = initial_rd
        self.teams: dict[str, GlickoTeam] = {}
        self.history: list = []

    def get_team(self, team: str) -> GlickoTeam:
        if team not in self.teams:
            self.teams[team] = GlickoTeam(rating=self.initial_rating, rd=self.initial_rd)
        return self.teams[team]

    def g(self, rd: float) -> float:
        """Glicko g-function."""
        return 1.0 / math.sqrt(1.0 + 3.0 * (rd / self.SCALE) ** 2 / (math.pi ** 2))

    def expected_score(self, rating_a: float, rd_a: float, rating_b: float, rd_b: float) -> float:
        """Expected score for team A against team B."""
        g_b = self.g(rd_b)
        return 1.0 / (1.0 + 10.0 ** (-g_b * (rating_a - rating_b) / (400.0)))

    def update_period(self, games: pd.DataFrame) -> None:
        """Update all teams after a rating period."""
        # Group games by team
        team_results = {team: [] for team in self.teams}
        for _, game in games.iterrows():
            home = game["HOME_TEAM"]
            away = game["AWAY_TEAM"]
            home_score = game["HOME_SCORE"]
            away_score = game["AWAY_SCORE"]
            # Home advantage adjustment
            home_team = self.get_team(home)
            away_team = self.get_team(away)
            home_result = 1.0 if home_score > away_score else (0.0 if home_score < away_score else 0.5)
            away_result = 1.0 - home_result
            team_results.setdefault(home, []).append((away, home_result))
            team_results.setdefault(away, []).append((home, away_result))

        # Update each team
        new_teams = {}
        for team in set(list(self.teams.keys()) + list(team_results.keys())):
            old = self.get_team(team)
            results = team_results.get(team, [])

            if not results:
                # RD increases for inactive teams
                new_rd = min(math.sqrt(old.rd ** 2 + self.c ** 2), 350.0)
                new_teams[team] = GlickoTeam(rating=old.rating, rd=new_rd, vol=old.vol)
                continue

            d_sq_inv = 0.0
            sum_g_sq_e = 0.0
            improvement = 0.0

            for opp_name, score in results:
                opp = self.get_team(opp_name)
                g_val = self.g(opp.rd)
                e_val = self.expected_score(old.rating, old.rd, opp.rating, opp.rd)
                d_sq_inv += g_val ** 2 * e_val * (1 - e_val)
                improvement += g_val * (score - e_val)

            if d_sq_inv == 0:
                new_teams[team] = GlickoTeam(rating=old.rating, rd=old.rd, vol=old.vol)
                continue

            d_sq = 1.0 / d_sq_inv
            new_rd = min(math.sqrt(1.0 / (1.0 / old.rd ** 2 + 1.0 / d_sq)), 350.0)
            new_rating = old.rating + d_sq * improvement
            new_teams[team] = GlickoTeam(rating=new_rating, rd=new_rd, vol=old.vol)

        self.history.append(dict(self.teams))
        self.teams = new_teams

    def win_probability(self, home: str, away: str, home_adv: float = 65.0) -> tuple[float, float]:
        """Calculate home win probability and uncertainty.

        Returns:
            Tuple of (win_prob, uncertainty) where uncertainty comes from RD.
        """
        h = self.get_team(home)
        a = self.get_team(away)
        adjusted_home = h.rating + home_adv
        prob = self.expected_score(adjusted_home, h.rd, a.rating, a.rd)
        combined_rd = math.sqrt(h.rd ** 2 + a.rd ** 2)
        return prob, combined_rd / 400.0  # Normalized uncertainty

print("GlickoRating class defined.")

In [ ]:
# Cell 21: Run Glicko on synthetic data (period-by-period)
#
# We process games in batches of 20 as "rating periods".
# Between periods, inactive teams' RD grows.

glicko = GlickoRating(c=63.2, initial_rating=1500, initial_rd=350)

# Process in rating periods of 20 games
period_size = 20
n_periods = len(games_df) // period_size

for i in range(n_periods):
    period_games = games_df.iloc[i * period_size : (i + 1) * period_size]
    glicko.update_period(period_games)

print("Glicko Ratings (after all periods):")
print(f"{'Team':<6} {'Rating':>8} {'RD':>8} {'True':>8}")
print(f"{'-'*6} {'-'*8} {'-'*8} {'-'*8}")
for team in sorted(TEAMS, key=lambda t: glicko.get_team(t).rating, reverse=True):
    t = glicko.get_team(team)
    print(f"{team:<6} {t.rating:>8.1f} {t.rd:>8.1f} {TRUE_STRENGTH[team]:>8.1f}")

In [ ]:
# Cell 22: Plot Glicko rating and RD evolution

glicko2 = GlickoRating(c=63.2, initial_rating=1500, initial_rd=350)
glicko2.update_period(games_df.iloc[:20])

rd_track = {team: [glicko2.get_team(team).rd] for team in TEAMS}
rating_track = {team: [glicko2.get_team(team).rating] for team in TEAMS}

for i in range(1, n_periods):
    period_games = games_df.iloc[i * period_size : (i + 1) * period_size]
    glicko2.update_period(period_games)
    for team in TEAMS:
        rd_track[team].append(glicko2.get_team(team).rd)
        rating_track[team].append(glicko2.get_team(team).rating)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Rating evolution
for team in TEAMS:
    ax1.plot(rating_track[team], label=team, linewidth=1.5, alpha=0.8)
ax1.set_xlabel("Rating Period", fontsize=12)
ax1.set_ylabel("Glicko Rating", fontsize=12)
ax1.set_title("Glicko Rating Evolution", fontsize=14, fontweight="bold")
ax1.legend(loc="upper left", fontsize=8, ncol=2)
ax1.grid(True, alpha=0.3)

# RD evolution
for team in TEAMS:
    ax2.plot(rd_track[team], label=team, linewidth=1.5, alpha=0.8)
ax2.set_xlabel("Rating Period", fontsize=12)
ax2.set_ylabel("Rating Deviation (RD)", fontsize=12)
ax2.set_title("Glicko RD Evolution (Uncertainty)", fontsize=14, fontweight="bold")
ax2.legend(loc="upper right", fontsize=8, ncol=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Section 8: Compare All Four Rating Systems

Now let's compare all four rating systems side by side. We normalize each system to a common scale (z-scores) so we can see which teams are consistently rated high or low.

In [ ]:
# Cell 24: Compare all four rating systems

comparison = {}
for team in TEAMS:
    elo_r = elo.get_rating(team)
    massey_r = massey_ratings.loc[team, "overall_rating"]
    pr_r = pr_ratings[team]
    glicko_r = glicko.get_team(team).rating
    glicko_rd = glicko.get_team(team).rd
    comparison[team] = {
        "Elo": elo_r,
        "Massey": massey_r,
        "PageRank": pr_r,
        "Glicko": glicko_r,
        "Glicko_RD": glicko_rd,
        "True": TRUE_STRENGTH[team],
    }

comp_df = pd.DataFrame(comparison).T

# Normalize to z-scores for comparison
for col in ["Elo", "Massey", "PageRank", "Glicko", "True"]:
    comp_df[f"{col}_z"] = (comp_df[col] - comp_df[col].mean()) / comp_df[col].std()

print("Rating Comparison (z-scored):")
print(f"{'Team':<6} {'Elo_z':>8} {'Massey_z':>8} {'PR_z':>8} {'Glicko_z':>8} {'True_z':>8}")
print(f"{'-'*6} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
for team in sorted(TEAMS, key=lambda t: comp_df.loc[t, "True_z"], reverse=True):
    row = comp_df.loc[team]
    print(f"{team:<6} {row['Elo_z']:>8.2f} {row['Massey_z']:>8.2f} {row['PageRank_z']:>8.2f} {row['Glicko_z']:>8.2f} {row['True_z']:>8.2f}")

In [ ]:
# Cell 25: Predict next-game win probabilities for all systems
#
# For a hypothetical matchup: KC (home) vs CAR (away)
# Each system predicts different win probabilities.

home_team = "KC"
away_team = "CAR"

# Elo prediction
elo_prob = elo.win_probability(home_team, away_team)

# Massey prediction (rating difference)
massey_diff = massey_ratings.loc[home_team, "overall_rating"] - massey_ratings.loc[away_team, "overall_rating"]
massey_prob = 1.0 / (1.0 + 10.0 ** (-massey_diff / 14.0))  # Convert spread to prob

# PageRank prediction
pr_diff = pr_ratings[home_team] - pr_ratings[away_team]
pr_prob = 1.0 / (1.0 + 10.0 ** (-pr_diff * 500 / 400.0))  # Scale PageRank diff

# Glicko prediction
glicko_prob, glicko_unc = glicko.win_probability(home_team, away_team)

print(f"Predicted Win Probabilities: {home_team} (home) vs {away_team} (away)")
print(f"{'System':<12} {'Home Win %':>12} {'Confidence':>12}")
print(f"{'-'*12} {'-'*12} {'-'*12}")
print(f"{'Elo':<12} {elo_prob:>11.1%} {'High':>12}")
print(f"{'Massey':<12} {massey_prob:>11.1%} {'High':>12}")
print(f"{'PageRank':<12} {pr_prob:>11.1%} {'Medium':>12}")
print(f"{'Glicko':<12} {glicko_prob:>11.1%} {f'±{glicko_unc:.2f}':>12}")
print(f"\nExpected spread (Massey): KC by {massey_diff:.1f} points")

In [ ]:
# Cell 26: Visual comparison of all four rating systems

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Elo
teams_sorted_elo = sorted(TEAMS, key=lambda t: elo.get_rating(t), reverse=True)
elo_vals = [elo.get_rating(t) for t in teams_sorted_elo]
axes[0, 0].bar(teams_sorted_elo, elo_vals, color='#2E86AB', alpha=0.8)
axes[0, 0].set_title('Elo Ratings', fontsize=13, fontweight='bold')
axes[0, 0].set_ylabel('Rating')
axes[0, 0].axhline(y=1500, color='gray', linestyle='--', alpha=0.5)

# Massey
teams_sorted_massey = sorted(TEAMS, key=lambda t: massey_ratings.loc[t, "overall_rating"], reverse=True)
massey_vals = [massey_ratings.loc[t, "overall_rating"] for t in teams_sorted_massey]
axes[0, 1].bar(teams_sorted_massey, massey_vals, color='#E63946', alpha=0.8)
axes[0, 1].set_title('Massey Ratings', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Rating')
axes[0, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# PageRank
teams_sorted_pr = sorted(TEAMS, key=lambda t: pr_ratings[t], reverse=True)
pr_vals = [pr_ratings[t] for t in teams_sorted_pr]
axes[1, 0].bar(teams_sorted_pr, pr_vals, color='#2A9D8F', alpha=0.8)
axes[1, 0].set_title('PageRank Ratings', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Rating')

# Glicko (with error bars)
teams_sorted_gl = sorted(TEAMS, key=lambda t: glicko.get_team(t).rating, reverse=True)
gl_ratings = [glicko.get_team(t).rating for t in teams_sorted_gl]
gl_rds = [glicko.get_team(t).rd for t in teams_sorted_gl]
axes[1, 1].bar(teams_sorted_gl, gl_ratings, color='#F4A261', alpha=0.8,
               yerr=gl_rds, capsize=3, ecolor='black', alpha=0.6)
axes[1, 1].set_title('Glicko Ratings (with RD error bars)', fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('Rating ± RD')

for ax in axes.flat:
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---

## Section 9: Backtest Each Rating System

A rating system is only useful if it predicts future games. Let's backtest each system by training on the first 75% of games and testing on the last 25%.

In [ ]:
# Cell 28: Backtest all four rating systems

n = len(games_df)
train_size = int(n * 0.75)
train_df = games_df.iloc[:train_size]
test_df = games_df.iloc[train_size:]

def backtest_elo(train: pd.DataFrame, test: pd.DataFrame, k: float = 32) -> dict:
    """Backtest Elo ratings."""
    model = EloRating(K=k, initial_rating=1500, home_advantage=65)
    for _, game in train.iterrows():
        model.update(game["HOME_TEAM"], game["AWAY_TEAM"], game["HOME_SCORE"], game["AWAY_SCORE"])
    correct = 0
    total = 0
    brier = []
    for _, game in test.iterrows():
        prob = model.win_probability(game["HOME_TEAM"], game["AWAY_TEAM"])
        actual = 1 if game["HOME_SCORE"] > game["AWAY_SCORE"] else 0
        pred = 1 if prob > 0.5 else 0
        correct += (pred == actual)
        total += 1
        brier.append((prob - actual) ** 2)
    return {"accuracy": correct / total, "brier": np.mean(brier), "total": total}

def backtest_massey(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    """Backtest Massey ratings."""
    model = MasseyRatings(home_advantage=3.0)
    ratings = model.compute_ratings(train)
    correct = 0
    total = 0
    brier = []
    for _, game in test.iterrows():
        h = game["HOME_TEAM"]
        a = game["AWAY_TEAM"]
        if h in ratings.index and a in ratings.index:
            diff = ratings.loc[h, "overall_rating"] - ratings.loc[a, "overall_rating"] + 3.0
            prob = 1.0 / (1.0 + 10.0 ** (-diff / 14.0))
            actual = 1 if game["HOME_SCORE"] > game["AWAY_SCORE"] else 0
            pred = 1 if prob > 0.5 else 0
            correct += (pred == actual)
            total += 1
            brier.append((prob - actual) ** 2)
    return {"accuracy": correct / total if total > 0 else 0, "brier": np.mean(brier) if brier else 0, "total": total}

def backtest_pagerank(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    """Backtest PageRank ratings."""
    model = PageRankRatings(damping=0.85)
    teams = sorted(set(train["HOME_TEAM"].unique()) | set(train["AWAY_TEAM"].unique()))
    ratings = model.compute_ratings(train, teams)
    correct = 0
    total = 0
    brier = []
    for _, game in test.iterrows():
        h = game["HOME_TEAM"]
        a = game["AWAY_TEAM"]
        if h in ratings and a in ratings:
            diff = (ratings[h] - ratings[a]) * 500  # Scale factor
            prob = 1.0 / (1.0 + 10.0 ** (-diff / 400.0))
            actual = 1 if game["HOME_SCORE"] > game["AWAY_SCORE"] else 0
            pred = 1 if prob > 0.5 else 0
            correct += (pred == actual)
            total += 1
            brier.append((prob - actual) ** 2)
    return {"accuracy": correct / total if total > 0 else 0, "brier": np.mean(brier) if brier else 0, "total": total}

def backtest_glicko(train: pd.DataFrame, test: pd.DataFrame, period_size: int = 20) -> dict:
    """Backtest Glicko ratings."""
    model = GlickoRating(c=63.2, initial_rating=1500, initial_rd=350)
    n_periods = len(train) // period_size
    for i in range(n_periods):
        period = train.iloc[i * period_size : (i + 1) * period_size]
        model.update_period(period)
    correct = 0
    total = 0
    brier = []
    for _, game in test.iterrows():
        prob, _ = model.win_probability(game["HOME_TEAM"], game["AWAY_TEAM"])
        actual = 1 if game["HOME_SCORE"] > game["AWAY_SCORE"] else 0
        pred = 1 if prob > 0.5 else 0
        correct += (pred == actual)
        total += 1
        brier.append((prob - actual) ** 2)
    return {"accuracy": correct / total if total > 0 else 0, "brier": np.mean(brier) if brier else 0, "total": total}

# Run backtests
results = {
    "Elo (K=32)": backtest_elo(train_df, test_df, k=32),
    "Elo (K=20)": backtest_elo(train_df, test_df, k=20),
    "Massey": backtest_massey(train_df, test_df),
    "PageRank": backtest_pagerank(train_df, test_df),
    "Glicko": backtest_glicko(train_df, test_df),
}

print("Backtest Results (75/25 split):")
print(f"{'System':<15} {'Accuracy':>10} {'Brier':>10} {'Games':>8}")
print(f"{'-'*15} {'-'*10} {'-'*10} {'-'*8}")
for name, r in results.items():
    print(f"{name:<15} {r['accuracy']:>9.1%} {r['brier']:>10.4f} {r['total']:>8d}")

---

## Exercises

Try these on your own:

1. **Tune Elo K-factor** — Run backtests with K=10, K=20, K=32, K=50. Which produces the best Brier score? Plot accuracy vs K.

2. **Combine Elo + Massey** — Create an ensemble that averages Elo win probability with Massey-derived probability. Does the ensemble beat either system alone?

3. **Predict the Super Bowl** — Use your best rating system to predict the winner of the Super Bowl. How confident are you? What does Glicko's RD say about your uncertainty?

4. **PageRank personalization** — Use `personalize_by_results()` to create a team-specific PageRank vector. How does KC's personalized ranking differ from the global one?

5. **Conference adjustment** — Use `massey.conference_adjustment()` to normalize ratings across conferences. Does this improve backtest accuracy?

---

## Summary

In this lab you learned:

- **Elo**: Simple iterative update, K-factor controls reactivity
- **Massey**: Linear algebra approach, decomposes into offense/defense
- **PageRank**: Transitive strength propagation, captures win quality
- **Glicko**: Elo + uncertainty (RD), essential for small samples
- How to compare rating systems using z-scores and backtesting
- How to convert ratings to win probabilities for prediction

### Key API Reference

| Class | Module | Purpose |
|---|---|---|
| `MasseyRatings` | `sportsquant.models.ratings.massey_ratings` | Massey method |
| `MasseyRatingsConfig` | `sportsquant.models.ratings.massey_ratings` | Configuration |
| `PageRankRatings` | `sportsquant.models.ratings.pagerank_ratings` | PageRank method |

### Key Concepts

| Concept | Description |
|---|---|
| **Elo K-factor** | Controls rating volatility; higher K = more reactive |
| **Massey decomposition** | Splits rating into offensive and defensive |
| **PageRank damping** | Probability of following a win edge (0.85 default) |
| **Glicko RD** | Rating deviation — measures uncertainty |
| **Backtesting** | Train on past data, test on future to validate |

### Next Steps

Continue to **Lab 08: Live Workflow** to learn how to connect rating systems with the poller for end-to-end live betting analysis.

---

*This lab used `MasseyRatings` and `PageRankRatings` from `sportsquant.models.ratings`. Elo and Glicko were implemented inline as reference implementations for educational purposes.*